# LNP-MFGO Phase 2B — Graph Neural Network Encoder with Cross-Component Attention

---

## Scientific Context: What Phases 1–2A Taught Us

### The Descriptor Ceiling

Across Phases 1 and 2A, we established a fundamental result: **molecular descriptors (Mordred, ECFP, RDKit)**  
**cannot predict LNP physicochemical properties across laboratories.** Even after removing study-level batch  
effects through Z-normalisation, the best descriptor-based model achieves R² = +0.018 with Spearman ρ = 0.12  
on 39-fold leave-one-study-out cross-validation. This is barely above random.

This is not a hyperparameter or model-selection problem — Optuna searched 60 configurations per algorithm  
and chose extreme regularisation (XGBoost α = 8.75, LightGBM depth = 3), confirming the signal genuinely  
does not exist in the descriptor space.

### Why Descriptors Fail: A Structural Argument

The failure has clear chemical reasons that directly motivate our GNN architecture:

**1. Descriptors are computed for isolated molecules, not formulations.**  
Mordred/RDKit/ECFP descriptors were calculated from the ionizable lipid SMILES alone. But an LNP is a  
*multi-component self-assembled system* — its particle size depends on how all four lipids (ionizable,  
helper, sterol, PEG) pack together. The same ionizable lipid can produce particle sizes ranging from  
50 nm to 300 nm depending on the helper lipid (DOPE vs DSPC) and molar ratios. No single-molecule  
descriptor captures these **inter-component interactions**.

**2. Fixed descriptors discard task-relevant substructural information.**  
Mordred computes 1,613 predefined mathematical features of a molecular graph. These features are generic —  
they were designed for drug-likeness and ADMET prediction, not for self-assembly physics. A GNN, by  
contrast, learns *task-specific* atom-level features through message passing, potentially discovering  
which substructures (e.g., amine headgroups, ester linkages, tail saturation) matter for particle size.

**3. ECFP fingerprints suffer from catastrophic hash collisions at low diversity.**  
With only 259 unique ionizable lipids hashed into 2,048 bits (97.9% zero), the fingerprint space is too  
sparse and collision-prone to distinguish structurally similar lipids that differ in biophysically  
important ways (e.g., single vs double tail, ester vs amide linker).

### What Phase 2B Introduces

We address each failure mode with a targeted architectural innovation:

| Descriptor Limitation | GNN Solution | Architecture Module |
|---|---|---|
| Single-molecule only | Process all 4 lipid graphs simultaneously | Multi-component GIN encoder |
| No inter-lipid interactions | Attention across lipid embeddings | Cross-component Transformer |
| Fixed features | Learnable message passing | 3-layer GIN with learned aggregation |
| No lab context | Per-study learnable vector | Study embedding layer |

The key scientific question: **Can learned molecular representations from GNNs capture within-study**  
**structure → property relationships that predefined descriptors miss?**

### Paper Narrative Position

This notebook produces the **central result** of the paper:

> *"We demonstrate that descriptor-based models fail to generalise across LNP studies (R² ≈ 0, LOSO-CV),*  
> *while graph neural networks with cross-component attention achieve [R² = X, ρ = Y], representing the*  
> *first positive cross-study prediction of LNP physicochemical properties."*

Even a modest improvement (R² > 0.05, ρ > 0.25) would be meaningful and publishable, because it  
demonstrates that learned representations extract signal that 1,613 handcrafted descriptors cannot.

---

## Section 0: Environment Setup

Phase 2B requires RDKit for molecular graph construction.  
We implement a GIN (Graph Isomorphism Network) **from scratch in pure PyTorch** — no PyTorch Geometric  
dependency. This makes the code fully portable and easier to reproduce.

**Why GIN?** Xu et al. (2019) proved GIN is as powerful as the Weisfeiler-Lehman graph isomorphism  
test — the theoretical maximum for message-passing GNNs. It is simpler than D-MPNN yet equally  
expressive, and easier to train on small datasets.

In [1]:
# ============================================================
# 0.1 — Install Dependencies
# ============================================================
# Uncomment on first run:
# !pip install rdkit-pypi  # molecular graph construction
# torch should already be installed from Phase 1

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time, json, copy
from pathlib import Path
from collections import defaultdict

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import spearmanr, pearsonr

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    print(f'✅ CUDA: {torch.cuda.get_device_name(0)}')
else:
    DEVICE = torch.device('cpu')
    print('⚠️  CPU mode')

PROJECT_DIR = Path('.')
OUTPUT_DIR = PROJECT_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR = OUTPUT_DIR / 'figures'
FIGURE_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    'figure.figsize': (10, 6), 'figure.dpi': 150, 'savefig.dpi': 300,
    'font.size': 11, 'font.family': 'sans-serif', 'axes.grid': True, 'grid.alpha': 0.3,
})
COLORS = {'primary': '#2563EB', 'secondary': '#DC2626', 'tertiary': '#059669',
          'quaternary': '#D97706', 'purple': '#7C3AED'}
print('Setup complete ✅')

✅ CUDA: NVIDIA GeForce RTX 3050 Laptop GPU
Setup complete ✅


---
## Section 1: Molecular Graph Construction from SMILES

A molecular graph represents each atom as a node and each bond as an edge.  
We convert SMILES strings into graph tensors using RDKit:

**Atom features (9-dimensional per atom):**  
Atomic number (one-hot, 10 common elements), degree, formal charge, number of Hs,  
hybridisation (one-hot), aromaticity, and whether the atom is in a ring.

**Bond features (4-dimensional per bond):**  
Bond type (single/double/triple/aromatic one-hot), conjugation, ring membership.

Each LNP formulation produces **4 molecular graphs** (one per lipid component),  
which are later processed by separate GNN encoders.

In [2]:
# ============================================================
# 1.1 — Atom & Bond Featurisation Functions
# ============================================================
# These functions convert RDKit atom/bond objects into numerical
# feature vectors. The feature dimensionality is fixed and known
# ahead of time so we can pre-allocate tensors.

ATOM_FEATURES_DIM = 33  # total atom feature dimension
BOND_FEATURES_DIM = 6   # total bond feature dimension

ATOM_LIST = [6, 7, 8, 9, 15, 16, 17, 35, 53, 0]  # C,N,O,F,P,S,Cl,Br,I,Other
HYBRIDIZATION_LIST = [
    Chem.rdchem.HybridizationType.SP,
    Chem.rdchem.HybridizationType.SP2,
    Chem.rdchem.HybridizationType.SP3,
    Chem.rdchem.HybridizationType.SP3D,
    Chem.rdchem.HybridizationType.SP3D2,
]

def one_hot(val, allowed_set):
    """One-hot encode a value against an allowed set."""
    encoding = [0] * (len(allowed_set) + 1)  # +1 for 'other'
    if val in allowed_set:
        encoding[allowed_set.index(val)] = 1
    else:
        encoding[-1] = 1
    return encoding


def atom_features(atom):
    """Convert an RDKit atom to a feature vector."""
    features = []
    features += one_hot(atom.GetAtomicNum(), ATOM_LIST)       # 11 dims
    features += one_hot(atom.GetDegree(), [0,1,2,3,4,5])     # 7 dims
    features += one_hot(atom.GetFormalCharge(), [-1,0,1,2])   # 5 dims
    features += one_hot(atom.GetNumRadicalElectrons(), [0,1]) # 3 dims
    features += one_hot(atom.GetHybridization(), HYBRIDIZATION_LIST)  # 6 dims
    features += [atom.GetIsAromatic()]                        # 1 dim
    return features  # total = 33


def bond_features(bond):
    """Convert an RDKit bond to a feature vector."""
    bt = bond.GetBondType()
    features = [
        bt == Chem.rdchem.BondType.SINGLE,
        bt == Chem.rdchem.BondType.DOUBLE,
        bt == Chem.rdchem.BondType.TRIPLE,
        bt == Chem.rdchem.BondType.AROMATIC,
        bond.GetIsConjugated(),
        bond.IsInRing(),
    ]
    return [float(f) for f in features]  # 6 dims

print(f'Atom feature dim: {ATOM_FEATURES_DIM}')
print(f'Bond feature dim: {BOND_FEATURES_DIM}')
print('Featurisation functions ready ✅')

Atom feature dim: 33
Bond feature dim: 6
Featurisation functions ready ✅


In [3]:
# ============================================================
# 1.2 — SMILES → Graph Conversion
# ============================================================
# Each molecule becomes a dict with:
#   'x': atom feature matrix [num_atoms, ATOM_FEATURES_DIM]
#   'edge_index': adjacency list [2, num_edges] (both directions)
#   'edge_attr': bond features [num_edges, BOND_FEATURES_DIM]
#   'num_atoms': number of atoms

def smiles_to_graph(smiles):
    """Convert a SMILES string to a graph dict."""
    if pd.isna(smiles) or not isinstance(smiles, str) or len(smiles) < 3:
        return None
    
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    # Atom features
    atoms = mol.GetAtoms()
    n_atoms = len(atoms)
    if n_atoms == 0:
        return None
    
    x = [atom_features(a) for a in atoms]
    x = torch.FloatTensor(x)  # [n_atoms, 33]
    
    # Edge index and features (bidirectional)
    edge_indices = []
    edge_feats = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        edge_indices.extend([[i, j], [j, i]])  # both directions
        edge_feats.extend([bf, bf])
    
    if len(edge_indices) == 0:
        # Single atom molecule — add self-loop
        edge_index = torch.LongTensor([[0], [0]])
        edge_attr = torch.zeros(1, BOND_FEATURES_DIM)
    else:
        edge_index = torch.LongTensor(edge_indices).t().contiguous()  # [2, n_edges]
        edge_attr = torch.FloatTensor(edge_feats)  # [n_edges, 6]
    
    return {
        'x': x,
        'edge_index': edge_index,
        'edge_attr': edge_attr,
        'num_atoms': n_atoms,
    }

# Quick test
test_graph = smiles_to_graph('CCCCCCCCN(CCCCCCCC)CC(O)CC(O)CC(O)CC(O)CCCCCC')
if test_graph:
    print(f'Test: {test_graph["num_atoms"]} atoms, '
          f'{test_graph["edge_index"].shape[1]} edges, '
          f'x shape: {test_graph["x"].shape}')
print('Graph conversion ready ✅')

Test: 35 atoms, 68 edges, x shape: torch.Size([35, 33])
Graph conversion ready ✅


In [4]:
# ============================================================
# 1.3 — Build Graphs for All LNP Formulations
# ============================================================
# Load data, compute study z-normalised targets, and convert
# all SMILES to molecular graphs. Rows where ANY lipid graph
# fails to parse are dropped.

df_raw = pd.read_csv(
    PROJECT_DIR / 'LNP_Atlas_Bioactivity_Reextracted.csv',
    encoding='latin-1', low_memory=False
)
print(f'Loaded: {df_raw.shape}')

# --- Re-compute Z-normalisation (self-contained) ---
TARGET_COL = 'particle_size_nm_std_num'
mask_target = df_raw[TARGET_COL].notna()

study_stats = df_raw[mask_target].groupby('paper_doi')[TARGET_COL].agg(['mean', 'std'])
global_mean = df_raw[mask_target][TARGET_COL].mean()
global_std = df_raw[mask_target][TARGET_COL].std()
study_stats['std'] = study_stats['std'].fillna(global_std)
study_stats.loc[study_stats['std'] == 0, 'std'] = global_std

df_raw['study_mean'] = df_raw['paper_doi'].map(study_stats['mean']).fillna(global_mean)
df_raw['study_std'] = df_raw['paper_doi'].map(study_stats['std']).fillna(global_std)
df_raw['target_znorm'] = (df_raw[TARGET_COL] - df_raw['study_mean']) / df_raw['study_std']

# --- Build molecular graphs ---
LIPID_SMILES_COLS = [
    'ionizable_lipid_smiles', 'helper_lipid_smiles',
    'sterol_lipid_smiles', 'peg_lipid_smiles'
]
LIPID_NAMES = ['ionizable', 'helper', 'sterol', 'peg']

# Cache: same SMILES → same graph (avoid recomputation)
graph_cache = {}

def get_graph(smiles):
    if smiles not in graph_cache:
        graph_cache[smiles] = smiles_to_graph(smiles)
    return graph_cache[smiles]

# Process all rows
print('Building molecular graphs...')
all_graphs = {name: [] for name in LIPID_NAMES}
valid_mask = []

for idx, row in df_raw.iterrows():
    row_ok = True
    row_graphs = {}
    for lipid_name, smiles_col in zip(LIPID_NAMES, LIPID_SMILES_COLS):
        smiles = row[smiles_col]
        if pd.isna(smiles) or not isinstance(smiles, str):
            row_ok = False
            break
        g = get_graph(smiles)
        if g is None:
            row_ok = False
            break
        row_graphs[lipid_name] = g
    
    if row_ok:
        for name in LIPID_NAMES:
            all_graphs[name].append(row_graphs[name])
        valid_mask.append(idx)

print(f'Valid formulations (all 4 SMILES parseable): {len(valid_mask)}/{len(df_raw)}')
print(f'Unique cached graphs: {len(graph_cache)}')

# Filter to valid rows
df = df_raw.loc[valid_mask].reset_index(drop=True)
print(f'\nWith particle size target: {df[TARGET_COL].notna().sum()}')
print(f'With z-norm target: {df["target_znorm"].notna().sum()}')
print(f'Studies: {df["paper_doi"].nunique()}')

Loaded: (934, 3730)
Building molecular graphs...
Valid formulations (all 4 SMILES parseable): 747/934
Unique cached graphs: 309

With particle size target: 717
With z-norm target: 717
Studies: 55


[12:26:10] Explicit valence for atom # 31 C, 5, is greater than permitted


In [5]:
# ============================================================
# 1.4 — Prepare Tabular Features (Ratios + Process)
# ============================================================
# The GNN encodes molecular structure. We also feed formulation-
# level features (molar ratios, process params) through a
# separate MLP branch, then fuse with the graph embeddings.

TABULAR_COLS = [
    'Ratio_Ionizable', 'Ratio_Helper', 'Ratio_Sterol', 'Ratio_PEG',
    'Process_FlowRate', 'Process_Ratio_AqOrg', 'Process_Is_Microfluidic', 'Process_pH',
]

# Encode study IDs for study embedding
study_encoder = LabelEncoder()
df['study_id'] = study_encoder.fit_transform(df['paper_doi'].fillna('Unknown'))
N_STUDIES = df['study_id'].nunique()

# Prepare tabular features
tabular_data = df[TABULAR_COLS].fillna(df[TABULAR_COLS].median()).values.astype(np.float32)

# Scale tabular features (fit on entire dataset — will refit per split later)
print(f'Tabular features: {tabular_data.shape}')
print(f'Study IDs: {N_STUDIES} unique')
print(f'Max atoms in a graph: {max(g["num_atoms"] for graphs in all_graphs.values() for g in graphs)}')
print('Tabular features ready ✅')

Tabular features: (747, 8)
Study IDs: 56 unique
Max atoms in a graph: 327
Tabular features ready ✅


---
## Section 2: GNN Architecture — Multi-Component GIN with Cross-Attention

The architecture has **four modules**, each addressing a specific limitation  
identified in Phases 1–2A:

**Module A: GIN Encoder (per lipid component)**  
A 3-layer Graph Isomorphism Network processes each lipid's molecular graph independently.  
GIN uses the update rule: `h_v^{(k+1) }= MLP^{(k)}((1 + ε) · h_v^{(k)} + Σ h_u^{(k)})`,  
which is provably as powerful as the WL graph isomorphism test.  
Global mean pooling aggregates atom embeddings into a fixed-size graph embedding.

**Module B: Cross-Component Attention**  
A small Transformer layer (1 head) attends across the 4 lipid embeddings.  
This allows the model to learn interactions — e.g., "this ionizable lipid pairs  
poorly with DSPC but well with DOPE." The output is a formulation-level embedding.

**Module C: Study Embedding**  
A learnable embedding vector per study (paper DOI). At test time for an  
unseen study, we use the mean of all training study embeddings.  
This conditions the model on the lab context without leaking target information.

**Module D: Prediction Head**  
A 2-layer MLP maps the fused embedding to the Z-normalised target.

In [6]:
# ============================================================
# 2.1 — GIN Layer (Graph Isomorphism Network)
# ============================================================
# Implements a single GIN message-passing layer.
# This is the core building block: each layer aggregates
# neighbour features and updates node representations.

class GINLayer(nn.Module):
    """Single GIN convolution layer."""
    def __init__(self, in_dim, out_dim, eps=0.0):
        super().__init__()
        self.eps = nn.Parameter(torch.tensor(eps))  # learnable epsilon
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
            nn.ReLU(),
        )
    
    def forward(self, x, edge_index, batch_num_atoms):
        """
        Args:
            x: [total_atoms, in_dim] — atom features for entire batch
            edge_index: [2, total_edges] — global edge indices
            batch_num_atoms: list of atom counts per graph (for bookkeeping)
        Returns:
            x_new: [total_atoms, out_dim]
        """
        # Aggregate neighbour features using scatter_add
        row, col = edge_index  # row = source, col = target
        agg = torch.zeros_like(x)
        agg.index_add_(0, col, x[row])  # sum of neighbour features
        
        # GIN update: (1 + eps) * h_self + sum(h_neighbours)
        out = (1 + self.eps) * x + agg
        return self.mlp(out)


class GINEncoder(nn.Module):
    """3-layer GIN encoder → global mean pooling → graph embedding."""
    def __init__(self, atom_dim=ATOM_FEATURES_DIM, hidden_dim=128, out_dim=128,
                 n_layers=3, dropout=0.2):
        super().__init__()
        self.atom_embed = nn.Linear(atom_dim, hidden_dim)
        self.layers = nn.ModuleList()
        for i in range(n_layers):
            self.layers.append(GINLayer(hidden_dim, hidden_dim))
        self.dropout = nn.Dropout(dropout)
        self.out_proj = nn.Linear(hidden_dim, out_dim)
    
    def forward(self, x, edge_index, batch_num_atoms):
        """
        Args:
            x: [total_atoms, atom_dim]
            edge_index: [2, total_edges]
            batch_num_atoms: [batch_size] list of atom counts
        Returns:
            graph_emb: [batch_size, out_dim]
        """
        h = self.atom_embed(x)
        for layer in self.layers:
            h = layer(h, edge_index, batch_num_atoms)
            h = self.dropout(h)
        
        # Global mean pooling per graph
        graph_embs = []
        offset = 0
        for n_atoms in batch_num_atoms:
            graph_embs.append(h[offset:offset+n_atoms].mean(dim=0))
            offset += n_atoms
        graph_emb = torch.stack(graph_embs)  # [batch_size, hidden_dim]
        return self.out_proj(graph_emb)

print(f'GINEncoder: {ATOM_FEATURES_DIM}→128→128, 3 layers')
print('GIN architecture defined ✅')

GINEncoder: 33→128→128, 3 layers
GIN architecture defined ✅


In [7]:
# ============================================================
# 2.2 — Cross-Component Attention + Full Model
# ============================================================
# After each lipid is encoded by GIN, we use a small Transformer
# attention layer to model inter-component interactions.

class CrossComponentAttention(nn.Module):
    """1-layer Transformer encoder over 4 lipid embeddings."""
    def __init__(self, embed_dim=128, n_heads=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim, n_heads, dropout=dropout, batch_first=True
        )
        self.norm = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 2, embed_dim),
        )
        self.norm2 = nn.LayerNorm(embed_dim)
    
    def forward(self, lipid_embeds):
        """
        Args:
            lipid_embeds: [batch, 4, embed_dim] — 4 lipid graph embeddings
        Returns:
            formulation_emb: [batch, embed_dim] — fused formulation embedding
            attn_weights: [batch, 4, 4] — attention matrix (for interpretability)
        """
        # Self-attention across 4 lipids
        attn_out, attn_weights = self.attn(
            lipid_embeds, lipid_embeds, lipid_embeds
        )
        x = self.norm(lipid_embeds + attn_out)  # residual
        x = self.norm2(x + self.ffn(x))          # FFN + residual
        
        # Pool across 4 components → single formulation embedding
        formulation_emb = x.mean(dim=1)  # [batch, embed_dim]
        return formulation_emb, attn_weights


class LNPMFGO(nn.Module):
    """
    LNP-MFGO: Multi-component GIN + Cross-Attention + Study Embedding.
    
    The full architecture for LNP property prediction.
    """
    def __init__(self, atom_dim=ATOM_FEATURES_DIM, hidden_dim=128, n_gin_layers=3,
                 tabular_dim=8, n_studies=100, study_emb_dim=16, dropout=0.2):
        super().__init__()
        
        # Module A: Separate GIN encoder per lipid type
        self.gin_encoders = nn.ModuleDict({
            'ionizable': GINEncoder(atom_dim, hidden_dim, hidden_dim, n_gin_layers, dropout),
            'helper': GINEncoder(atom_dim, hidden_dim, hidden_dim, n_gin_layers, dropout),
            'sterol': GINEncoder(atom_dim, hidden_dim, hidden_dim, n_gin_layers, dropout),
            'peg': GINEncoder(atom_dim, hidden_dim, hidden_dim, n_gin_layers, dropout),
        })
        
        # Module B: Cross-component attention
        self.cross_attn = CrossComponentAttention(hidden_dim, n_heads=4, dropout=dropout)
        
        # Module C: Study embedding
        self.study_embedding = nn.Embedding(n_studies + 1, study_emb_dim)  # +1 for unknown
        
        # Tabular feature encoder
        self.tabular_encoder = nn.Sequential(
            nn.Linear(tabular_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 32),
        )
        
        # Module D: Prediction head
        fusion_dim = hidden_dim + 32 + study_emb_dim  # graph + tabular + study
        self.predictor = nn.Sequential(
            nn.Linear(fusion_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )
    
    def forward(self, graph_batch, tabular, study_ids):
        """
        Args:
            graph_batch: dict with keys 'ionizable','helper','sterol','peg',
                         each containing (x, edge_index, batch_num_atoms)
            tabular: [batch, tabular_dim]
            study_ids: [batch] LongTensor
        Returns:
            pred: [batch] predictions
            attn_weights: [batch, 4, 4] cross-component attention
        """
        # Encode each lipid component
        lipid_embs = []
        for name in ['ionizable', 'helper', 'sterol', 'peg']:
            x, ei, bna = graph_batch[name]
            emb = self.gin_encoders[name](x, ei, bna)
            lipid_embs.append(emb)
        
        # Stack → [batch, 4, hidden_dim]
        lipid_stack = torch.stack(lipid_embs, dim=1)
        
        # Cross-component attention
        formulation_emb, attn_weights = self.cross_attn(lipid_stack)
        
        # Encode tabular features
        tab_emb = self.tabular_encoder(tabular)
        
        # Study embedding
        study_emb = self.study_embedding(study_ids)
        
        # Fuse all
        fused = torch.cat([formulation_emb, tab_emb, study_emb], dim=-1)
        pred = self.predictor(fused).squeeze(-1)
        
        return pred, attn_weights


# Count parameters
model_test = LNPMFGO(n_studies=N_STUDIES)
n_params = sum(p.numel() for p in model_test.parameters())
print(f'LNP-MFGO total parameters: {n_params:,}')
print(f'  4 × GIN encoder + Cross-Attention + Study Embedding + Predictor')
del model_test
print('Model architecture defined ✅')

LNP-MFGO total parameters: 628,957
  4 × GIN encoder + Cross-Attention + Study Embedding + Predictor
Model architecture defined ✅


---
## Section 3: Dataset, Collation & Training Loop

Graph data requires special batching: we concatenate all atom feature matrices  
and offset edge indices so that the entire batch becomes one large disconnected  
graph. This is the standard approach in graph ML and is memory-efficient.

The training loop uses:
- **MSE loss** on Z-normalised targets
- **AdamW optimiser** with weight decay (regularisation)
- **Cosine annealing** learning rate schedule
- **Early stopping** on validation loss (patience=30 epochs)
- **Gradient clipping** to prevent exploding gradients in GNNs

In [8]:
# ============================================================
# 3.1 — Custom Dataset & Collate Function
# ============================================================

class LNPGraphDataset(Dataset):
    """Dataset that returns graph dicts + tabular features + target."""
    def __init__(self, indices, graphs_dict, tabular, targets, study_ids):
        self.indices = indices
        self.graphs_dict = graphs_dict  # {'ionizable': [...], 'helper': [...], ...}
        self.tabular = tabular
        self.targets = targets
        self.study_ids = study_ids
    
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        i = self.indices[idx]
        graphs = {name: self.graphs_dict[name][i] for name in LIPID_NAMES}
        return graphs, self.tabular[i], self.targets[i], self.study_ids[i]


def collate_lnp(batch):
    """
    Custom collate: batch molecular graphs by concatenating atoms
    and offsetting edge indices.
    """
    graphs_list, tab_list, target_list, study_list = zip(*batch)
    
    graph_batch = {}
    for name in LIPID_NAMES:
        all_x, all_ei, batch_num_atoms = [], [], []
        atom_offset = 0
        for sample_graphs in graphs_list:
            g = sample_graphs[name]
            all_x.append(g['x'])
            all_ei.append(g['edge_index'] + atom_offset)
            batch_num_atoms.append(g['num_atoms'])
            atom_offset += g['num_atoms']
        
        graph_batch[name] = (
            torch.cat(all_x, dim=0),           # [total_atoms, atom_dim]
            torch.cat(all_ei, dim=1),           # [2, total_edges]
            batch_num_atoms,                     # list of int
        )
    
    tabular = torch.FloatTensor(np.array(tab_list))
    targets = torch.FloatTensor(np.array(target_list))
    study_ids = torch.LongTensor(np.array(study_list))
    
    return graph_batch, tabular, targets, study_ids

print('Dataset and collation ready ✅')

Dataset and collation ready ✅


In [9]:
# ============================================================
# 3.2 — Training & Evaluation Functions
# ============================================================

def to_device(graph_batch, tabular, targets, study_ids, device):
    """Move all tensors to device."""
    gb = {}
    for name in LIPID_NAMES:
        x, ei, bna = graph_batch[name]
        gb[name] = (x.to(device), ei.to(device), bna)
    return gb, tabular.to(device), targets.to(device), study_ids.to(device)


def evaluate_metrics(y_true, y_pred):
    """Compute all regression metrics."""
    yt, yp = np.array(y_true), np.array(y_pred)
    mask = np.isfinite(yt) & np.isfinite(yp)
    yt, yp = yt[mask], yp[mask]
    if len(yt) < 3:
        return {'R2': np.nan, 'RMSE': np.nan, 'MAE': np.nan,
                'Pearson_r': np.nan, 'Spearman_rho': np.nan, 'N': len(yt)}
    return {
        'R2': r2_score(yt, yp),
        'RMSE': np.sqrt(mean_squared_error(yt, yp)),
        'MAE': mean_absolute_error(yt, yp),
        'Pearson_r': pearsonr(yt, yp)[0],
        'Spearman_rho': spearmanr(yt, yp)[0],
        'N': len(yt),
    }


def train_model(model, train_loader, val_loader, device,
                epochs=150, lr=5e-4, patience=30, weight_decay=1e-4):
    """Train with early stopping."""
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.MSELoss()
    
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    
    for epoch in range(epochs):
        # --- Train ---
        model.train()
        train_loss = 0
        n_batches = 0
        for graph_batch, tabular, targets, study_ids in train_loader:
            gb, tab, tgt, sid = to_device(graph_batch, tabular, targets, study_ids, device)
            optimizer.zero_grad()
            pred, _ = model(gb, tab, sid)
            loss = criterion(pred, tgt)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
            n_batches += 1
        scheduler.step()
        
        # --- Validate ---
        model.eval()
        val_loss = 0
        n_val = 0
        with torch.no_grad():
            for graph_batch, tabular, targets, study_ids in val_loader:
                gb, tab, tgt, sid = to_device(graph_batch, tabular, targets, study_ids, device)
                pred, _ = model(gb, tab, sid)
                val_loss += criterion(pred, tgt).item()
                n_val += 1
        val_loss /= max(n_val, 1)
        
        # --- Early stopping ---
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    
    model.load_state_dict(best_state)
    return model, epoch + 1


def predict(model, loader, device):
    """Get predictions from trained model."""
    model.eval()
    preds, trues = [], []
    attn_all = []
    with torch.no_grad():
        for graph_batch, tabular, targets, study_ids in loader:
            gb, tab, tgt, sid = to_device(graph_batch, tabular, targets, study_ids, device)
            pred, attn_w = model(gb, tab, sid)
            preds.append(pred.cpu().numpy())
            trues.append(tgt.cpu().numpy())
            attn_all.append(attn_w.cpu().numpy())
    return np.concatenate(preds), np.concatenate(trues), np.concatenate(attn_all)

print('Training functions ready ✅')

Training functions ready ✅


---
## Section 4: Leave-One-Study-Out Cross-Validation

We use the identical 39-fold LOSO-CV protocol from Phase 2A for fair comparison.  
For each fold:

1. Hold out one study as **test** set
2. Use 10% of remaining studies as **validation** (for early stopping)
3. Train GNN on the rest
4. For the held-out study's study embedding, use the **mean** of all training study embeddings  
   (since we've never seen this study before)

This is the most rigorous evaluation: the model must predict a lab it has never seen.

In [10]:
# ============================================================
# 4.1 — LOSO-CV Loop for GNN
# ============================================================
# This is the main experiment. We run all 39 LOSO folds.
# Expected runtime: 30-90 min on GPU, 3-8 hrs on CPU.

# Filter to rows with target
mask_t = df['target_znorm'].notna()
df_model = df[mask_t].reset_index(drop=True)

# Rebuild graph lists for filtered indices
valid_indices_target = df[mask_t].index.tolist()
graphs_filtered = {name: [all_graphs[name][i] for i in valid_indices_target]
                   for name in LIPID_NAMES}
tabular_filtered = tabular_data[valid_indices_target]
targets_filtered = df_model['target_znorm'].values.astype(np.float32)
study_ids_filtered = df_model['study_id'].values
studies_filtered = df_model['paper_doi']

# Identify usable studies (≥5 samples)
study_counts = studies_filtered.value_counts()
usable_studies = study_counts[study_counts >= 5].index.tolist()
print(f'Total samples: {len(df_model)}')
print(f'Usable studies for LOSO: {len(usable_studies)}')
print(f'Samples in usable studies: {studies_filtered.isin(usable_studies).sum()}')

# Tabular scaler (will refit per fold)
tab_scaler = StandardScaler()

BATCH_SIZE = 32
HIDDEN_DIM = 128
N_GIN_LAYERS = 3
LR = 5e-4
EPOCHS = 150
PATIENCE = 30

print(f'\nHyperparameters: hidden={HIDDEN_DIM}, layers={N_GIN_LAYERS}, '
      f'lr={LR}, batch={BATCH_SIZE}, patience={PATIENCE}')

Total samples: 717
Usable studies for LOSO: 36
Samples in usable studies: 672

Hyperparameters: hidden=128, layers=3, lr=0.0005, batch=32, patience=30


In [ ]:
# ============================================================
# 4.2 — Run LOSO-CV
# ============================================================

gnn_fold_results = []
gnn_all_preds = {}
gnn_all_attn = []

total_folds = len(usable_studies)
t_start = time.time()

for fold_idx, test_study in enumerate(usable_studies):
    # --- Split indices ---
    test_mask = (studies_filtered.values == test_study)
    train_val_mask = ~test_mask
    
    train_val_idx = np.where(train_val_mask)[0]
    test_idx = np.where(test_mask)[0]
    
    if len(test_idx) < 3 or len(train_val_idx) < 20:
        continue
    
    # Split train_val into train + val (90/10)
    np.random.seed(SEED + fold_idx)
    val_size = max(5, int(0.1 * len(train_val_idx)))
    perm = np.random.permutation(len(train_val_idx))
    val_local = train_val_idx[perm[:val_size]]
    train_local = train_val_idx[perm[val_size:]]
    
    # Scale tabular features
    tab_sc = StandardScaler()
    tab_train = tab_sc.fit_transform(tabular_filtered[train_local])
    tab_val = tab_sc.transform(tabular_filtered[val_local])
    tab_test = tab_sc.transform(tabular_filtered[test_idx])
    
    # Handle unseen study: use N_STUDIES as the "unknown" ID
    test_study_id_mapped = N_STUDIES  # last index = unknown
    study_ids_train = study_ids_filtered[train_local]
    study_ids_val = study_ids_filtered[val_local]
    study_ids_test = np.full(len(test_idx), test_study_id_mapped)
    
    # Create datasets
    train_ds = LNPGraphDataset(train_local, graphs_filtered, tab_train,
                               targets_filtered[train_local], study_ids_train)
    val_ds = LNPGraphDataset(val_local, graphs_filtered, tab_val,
                             targets_filtered[val_local], study_ids_val)
    test_ds = LNPGraphDataset(test_idx, graphs_filtered, tab_test,
                              targets_filtered[test_idx], study_ids_test)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=collate_lnp, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            collate_fn=collate_lnp)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                             collate_fn=collate_lnp)
    
    # --- Train ---
    model = LNPMFGO(
        atom_dim=ATOM_FEATURES_DIM, hidden_dim=HIDDEN_DIM,
        n_gin_layers=N_GIN_LAYERS, tabular_dim=len(TABULAR_COLS),
        n_studies=N_STUDIES, study_emb_dim=16, dropout=0.2
    ).to(DEVICE)
    
    # Initialise unknown study embedding as mean of all others
    with torch.no_grad():
        known_embs = model.study_embedding.weight[:N_STUDIES].mean(dim=0)
        model.study_embedding.weight[N_STUDIES] = known_embs
    
    # Rebuild split-specific datasets with local indexing (fixes out-of-bounds indexing)
    train_graphs = {name: [graphs_filtered[name][i] for i in train_local] for name in LIPID_NAMES}
    val_graphs   = {name: [graphs_filtered[name][i] for i in val_local] for name in LIPID_NAMES}
    test_graphs  = {name: [graphs_filtered[name][i] for i in test_idx] for name in LIPID_NAMES}

    train_ds = LNPGraphDataset(
        np.arange(len(train_local)), train_graphs, tab_train,
        targets_filtered[train_local], study_ids_train
    )
    val_ds = LNPGraphDataset(
        np.arange(len(val_local)), val_graphs, tab_val,
        targets_filtered[val_local], study_ids_val
    )
    test_ds = LNPGraphDataset(
        np.arange(len(test_idx)), test_graphs, tab_test,
        targets_filtered[test_idx], study_ids_test
    )

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=collate_lnp, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            collate_fn=collate_lnp)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                             collate_fn=collate_lnp)

    model, n_epochs = train_model(
        model, train_loader, val_loader, DEVICE,
        epochs=EPOCHS, lr=LR, patience=PATIENCE
    )
    
    # --- Evaluate ---
    # Update unknown embedding to mean of trained embeddings
    with torch.no_grad():
        unique_train_studies = np.unique(study_ids_train)
        mean_emb = model.study_embedding.weight[unique_train_studies].mean(dim=0)
        model.study_embedding.weight[N_STUDIES] = mean_emb
    
    pred, true, attn = predict(model, test_loader, DEVICE)
    metrics = evaluate_metrics(true, pred)
    metrics['study'] = test_study
    metrics['n_epochs'] = n_epochs
    metrics['train_size'] = len(train_local)
    gnn_fold_results.append(metrics)
    
    # Store predictions
    for idx_local, p in zip(test_idx, pred):
        gnn_all_preds[idx_local] = p
    gnn_all_attn.append(attn)
    
    elapsed = time.time() - t_start
    eta = elapsed / (fold_idx + 1) * (total_folds - fold_idx - 1)
    r2_str = f'{metrics["R2"]:+.4f}' if np.isfinite(metrics['R2']) else 'N/A'
    print(f'  Fold {fold_idx+1:2d}/{total_folds}  '
          f'Study={str(test_study).split("/")[-1][:25]:25s}  '
          f'R²={r2_str}  ρ={metrics["Spearman_rho"]:+.3f}  '
          f'({n_epochs}ep, ETA {eta/60:.0f}min)')
    
    # Free memory
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

total_time = time.time() - t_start
print(f'\n✅ LOSO-CV complete: {len(gnn_fold_results)} folds in {total_time/60:.1f} min')

  Fold  1/36  Study=s41467-024-45422-9         R²=-0.1105  ρ=-0.179  (62ep, ETA 92min)


---
## Section 5: Results — GNN vs Descriptor Baselines

The key comparison: does the GNN with cross-component attention outperform  
descriptor-based models on the identical LOSO-CV benchmark?

In [ ]:
# ============================================================
# 5.1 — GNN LOSO-CV Summary
# ============================================================

gnn_df = pd.DataFrame(gnn_fold_results)

valid_folds = gnn_df[gnn_df['R2'].notna()]
r2s = valid_folds['R2'].values
rhos = valid_folds['Spearman_rho'].values

print('GNN (LNP-MFGO) LOSO-CV RESULTS — Particle Size (Z-Normalised)')
print('=' * 70)
print(f'  Folds evaluated:      {len(valid_folds)}')
print(f'  Mean R²:              {np.mean(r2s):+.4f} ± {np.std(r2s):.4f}')
print(f'  Median R²:            {np.median(r2s):+.4f}')
print(f'  % Positive R² folds:  {100*np.mean(r2s > 0):.1f}%')
print(f'  Mean Spearman ρ:      {np.nanmean(rhos):+.4f} ± {np.nanstd(rhos):.4f}')
print(f'  Mean RMSE:            {valid_folds["RMSE"].mean():.4f}')
print(f'  Mean MAE:             {valid_folds["MAE"].mean():.4f}')

# Save
gnn_df.to_csv(OUTPUT_DIR / 'phase2b_gnn_fold_results.csv', index=False)
print(f'\nSaved: phase2b_gnn_fold_results.csv')

In [ ]:
# ============================================================
# 5.2 — Head-to-Head: GNN vs Phase 2A Baselines
# ============================================================
# Load Phase 2A results for comparison.

phase2a_file = OUTPUT_DIR / 'phase2a_loso_results.csv'
if phase2a_file.exists():
    p2a = pd.read_csv(phase2a_file)
    znorm_baselines = p2a[p2a['Normalised'].str.contains('Z-Norm')].copy()
else:
    # Hardcode Phase 2A results if file not available
    znorm_baselines = pd.DataFrame([
        {'Model': 'LightGBM (tuned)', 'Mean_R2': 0.0181, 'Mean_Spearman': 0.124, 'Pct_Positive_R2': 38.5},
        {'Model': 'XGBoost (tuned)', 'Mean_R2': 0.0154, 'Mean_Spearman': 0.180, 'Pct_Positive_R2': 46.2},
        {'Model': 'RF (tuned)', 'Mean_R2': 0.0019, 'Mean_Spearman': 0.178, 'Pct_Positive_R2': 51.3},
        {'Model': 'RF (default)', 'Mean_R2': 0.0098, 'Mean_Spearman': 0.105, 'Pct_Positive_R2': 46.2},
        {'Model': 'SVR (RBF)', 'Mean_R2': -0.2427, 'Mean_Spearman': 0.070, 'Pct_Positive_R2': 23.1},
    ])

# GNN summary row
gnn_summary = {
    'Model': 'GNN (LNP-MFGO)',
    'Mean_R2': np.mean(r2s),
    'Std_R2': np.std(r2s),
    'Mean_Spearman': np.nanmean(rhos),
    'Pct_Positive_R2': 100 * np.mean(r2s > 0),
    'N_folds': len(valid_folds),
}

print('HEAD-TO-HEAD COMPARISON (Particle Size, Z-Normalised, LOSO-CV)')
print('=' * 80)
print(f'{"Model":30s} {"Mean R²":>10s} {"Spearman ρ":>12s} {"% Positive":>10s}')
print('-' * 80)

# Print GNN first (highlighted)
print(f'  {">>> GNN (LNP-MFGO)":28s} {gnn_summary["Mean_R2"]:+10.4f} '
      f'{gnn_summary["Mean_Spearman"]:+12.4f} {gnn_summary["Pct_Positive_R2"]:9.1f}%')
print('-' * 80)

# Print baselines
for _, row in znorm_baselines.iterrows():
    model_name = row.get('Model', 'Unknown')
    mean_r2 = row.get('Mean_R2', np.nan)
    mean_sp = row.get('Mean_Spearman', np.nan)
    pct_pos = row.get('Pct_Positive_R2', np.nan)
    print(f'  {model_name:28s} {mean_r2:+10.4f} {mean_sp:+12.4f} {pct_pos:9.1f}%')

# Save comparison
comparison_rows = [gnn_summary]
for _, row in znorm_baselines.iterrows():
    comparison_rows.append(row.to_dict())
comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(OUTPUT_DIR / 'phase2b_comparison.csv', index=False)
print(f'\nSaved: phase2b_comparison.csv')

In [ ]:
# ============================================================
# 5.3 — Per-Fold Comparison Figure
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: R² per fold, sorted
ax = axes[0]
sorted_r2 = gnn_df.sort_values('R2', ascending=False)
colors_bar = [COLORS['tertiary'] if r > 0 else COLORS['secondary'] for r in sorted_r2['R2']]
ax.barh(range(len(sorted_r2)), sorted_r2['R2'].values, color=colors_bar, alpha=0.8)
ax.set_yticks(range(len(sorted_r2)))
ax.set_yticklabels([str(s).split('/')[-1][:20] for s in sorted_r2['study']], fontsize=7)
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Test R²')
ax.set_title(f'GNN LOSO-CV R² per Study\n'
             f'Mean={np.mean(r2s):+.4f}, {100*np.mean(r2s>0):.0f}% positive',
             fontweight='bold')
ax.invert_yaxis()

# Right: GNN R² vs Phase 2A best baseline R² (per fold)
ax = axes[1]
# We'll need Phase 2A fold details for direct comparison
p2a_detail_file = OUTPUT_DIR / 'phase2a_fold_details.csv'
if p2a_detail_file.exists():
    p2a_folds = pd.read_csv(p2a_detail_file)
    best_baseline_config = p2a_folds.groupby('Configuration')['R2'].mean().idxmax()
    baseline_folds = p2a_folds[p2a_folds['Configuration'] == best_baseline_config]
    
    # Match by study
    merged = gnn_df.merge(baseline_folds[['study', 'R2']], on='study',
                          suffixes=('_GNN', '_Baseline'), how='inner')
    if len(merged) > 5:
        ax.scatter(merged['R2_Baseline'], merged['R2_GNN'],
                   c=COLORS['primary'], s=50, alpha=0.7, edgecolors='white')
        lims = [min(merged[['R2_Baseline','R2_GNN']].min().min(), -0.2),
                max(merged[['R2_Baseline','R2_GNN']].max().max(), 0.2)]
        ax.plot(lims, lims, 'k--', alpha=0.5, label='Equal performance')
        ax.set_xlabel(f'Baseline R² ({best_baseline_config.split("|")[0].strip()})')
        ax.set_ylabel('GNN (LNP-MFGO) R²')
        above = (merged['R2_GNN'] > merged['R2_Baseline']).sum()
        ax.set_title(f'GNN vs Best Baseline (per fold)\n'
                     f'GNN wins {above}/{len(merged)} folds', fontweight='bold')
        ax.legend()
    else:
        ax.text(0.5, 0.5, 'Insufficient matched folds\nfor comparison',
                ha='center', va='center', transform=ax.transAxes)
else:
    ax.text(0.5, 0.5, 'Phase 2A fold details not found\nRun Phase 2A first',
            ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'fig11_gnn_loso_results.png', bbox_inches='tight')
plt.show()
print('Saved: fig11_gnn_loso_results.png')

In [ ]:
# ============================================================
# 5.4 — Cross-Component Attention Visualisation
# ============================================================
# This is a KEY figure for the paper: it shows which lipid
# components the model attends to most when predicting properties.

if gnn_all_attn:
    all_attn_np = np.concatenate(gnn_all_attn, axis=0)  # [N, 4, 4]
    mean_attn = all_attn_np.mean(axis=0)  # [4, 4]
    
    fig, ax = plt.subplots(figsize=(7, 6))
    labels = ['Ionizable', 'Helper', 'Sterol', 'PEG']
    sns.heatmap(mean_attn, xticklabels=labels, yticklabels=labels,
                annot=True, fmt='.3f', cmap='Blues', ax=ax,
                vmin=0, vmax=mean_attn.max() * 1.1)
    ax.set_title('Mean Cross-Component Attention Weights\n'
                 '(averaged across all LOSO test predictions)', fontweight='bold')
    ax.set_ylabel('Query (attending FROM)')
    ax.set_xlabel('Key (attending TO)')
    
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / 'fig12_cross_attention.png', bbox_inches='tight')
    plt.show()
    print('Saved: fig12_cross_attention.png')
    print(f'\nInterpretation: higher values mean the model attends more to that component.')
    print(f'Diagonal = self-attention; off-diagonal = cross-component interaction.')
else:
    print('No attention weights collected — check LOSO-CV results.')

In [ ]:
# ============================================================
# 5.5 — Save All Phase 2B Outputs
# ============================================================

# Save attention matrix
if gnn_all_attn:
    all_attn_np = np.concatenate(gnn_all_attn, axis=0)
    np.save(OUTPUT_DIR / 'cross_attention_weights.npy', all_attn_np)

print('PHASE 2B OUTPUTS:')
print(f'  phase2b_gnn_fold_results.csv    — Per-fold GNN LOSO results')
print(f'  phase2b_comparison.csv           — GNN vs all baselines')
print(f'  cross_attention_weights.npy      — Attention matrices for interpretability')
print(f'  figures/fig11_gnn_loso_results   — Per-fold R² + GNN vs baseline')
print(f'  figures/fig12_cross_attention     — Attention heatmap')

---
## Summary & What the Results Mean for the Paper

### Result Scenarios & Paper Framing

**Scenario A: GNN clearly beats baselines (R² > 0.05, ρ > 0.25)**  
→ *"Learned molecular representations capture structure-property relationships invisible to*  
*handcrafted descriptors, achieving the first positive cross-study LNP property prediction."*  
→ Proceed to Phase 2C (adversarial adaptation) + Phase 3 (multi-task + bioactivity)

**Scenario B: GNN matches baselines (R² ≈ 0, ρ ≈ 0.15)**  
→ *"Neither descriptors nor GNNs solve cross-study generalisation with current data,*  
*suggesting the bottleneck is measurement heterogeneity, not molecular representation."*  
→ Pivot to domain adaptation (Phase 2C) as the primary contribution  
→ Paper becomes a **benchmarking + methodology** contribution (still publishable)

**Scenario C: GNN worse than baselines**  
→ *"GNNs overfit on small, heterogeneous datasets — architectural solutions alone are*  
*insufficient without domain adaptation or data augmentation."*  
→ Important negative result + pivot to augmentation strategies

### Files to Upload for Phase 2C Planning
1. `phase2b_gnn_fold_results.csv`
2. `phase2b_comparison.csv`
3. `cross_attention_weights.npy` (optional but useful)